<h1>Tidy data</h1>
            
<small>Source: <a href="https://github.com/tidyverse/tidyr/blob/pkgdown-v1.3.1/vignettes/tidy-data.Rmd"><code>vignettes/tidy-data.Rmd</code></a></small>
<div><code>tidy-data.Rmd</code></div>


<p>(This is an informal and code heavy version of the full <a href="https://vita.had.co.nz/papers/tidy-data.html">tidy data paper</a>. Please refer to that for more details.)</p>

# <h2 id="data-tidying">Data tidying<a href="#data-tidying"></a></h2>

<p>It is often said that 80% of data analysis is spent on the cleaning and preparing data. And it’s not just a first step, but it must be repeated many times over the course of analysis as new problems come to light or new data is collected. To get a handle on the problem, this paper focuses on a small, but important, aspect of data cleaning that I call data <strong>tidying</strong>: structuring datasets to facilitate analysis.</p>

<p>The principles of tidy data provide a standard way to organise data values within a dataset. A standard makes initial data cleaning easier because you don’t need to start from scratch and reinvent the wheel every time. The tidy data standard has been designed to facilitate initial exploration and analysis of the data, and to simplify the development of data analysis tools that work well together. Current tools often require translation. You have to spend time munging the output from one tool so you can input it into another. Tidy datasets and tidy tools work hand in hand to make data analysis easier, allowing you to focus on the interesting domain problem, not on the uninteresting logistics of data.</p>

# <h2 id="defining">Defining tidy data</h2>

<blockquote>
<p>Happy families are all alike; every unhappy family is unhappy in its own way — Leo Tolstoy</p>
</blockquote>

<p>Like families, tidy datasets are all alike but every messy dataset is messy in its own way. Tidy datasets provide a standardized way to link the structure of a dataset (its physical layout) with its semantics (its meaning). In this section, I’ll provide some standard vocabulary for describing the structure and semantics of a dataset, and then use those definitions to define tidy data.</p>


## <h3 id="data-structure">Data structure</h3>

<p>Most statistical datasets are data frames made up of <strong>rows</strong> and <strong>columns</strong>. The columns are almost always labeled and the rows are sometimes labeled. The following code provides some data about an imaginary classroom in a format commonly seen in the wild. The table has three columns and four rows, and both rows and columns are labeled.</p>

In [2]:
library(tibble)

classroom <- tribble(
  ~name,    ~quiz1, ~quiz2, ~test1,
  "Billy",  NA,     "D",    "C",
  "Suzy",   "F",    NA,     NA,
  "Lionel", "B",    "C",    "B",
  "Jenny",  "A",    "A",    "B"
  )

classroom

name,quiz1,quiz2,test1
<chr>,<chr>,<chr>,<chr>
Billy,NA,D,C
Suzy,F,NA,NA
Lionel,B,C,B
Jenny,A,A,B


<p>There are many ways to structure the same underlying data. The following table shows the same data as above, but the rows and columns have been transposed.</p>

In [3]:
tribble(
  ~assessment, ~Billy, ~Suzy, ~Lionel, ~Jenny,
  "quiz1",     NA,     "F",   "B",     "A",
  "quiz2",     "D",    NA,    "C",     "A",
  "test1",     "C",    NA,    "B",     "B"
  )

assessment,Billy,Suzy,Lionel,Jenny
<chr>,<chr>,<chr>,<chr>,<chr>
quiz1,NA,F,B,A
quiz2,D,NA,C,A
test1,C,NA,B,B


<p>The data is the same, but the layout is different. Our vocabulary of rows and columns is simply not rich enough to describe why the two tables represent the same data. In addition to appearance, we need a way to describe the underlying semantics, or meaning, of the values displayed in the table.</p>

## <h3 id="data-semantics">Data semantics</h3>

<p>A dataset is a collection of <strong>values</strong>, usually either numbers (if quantitative) or strings (if qualitative). Values are organised in two ways. Every value belongs to a <strong>variable</strong> and an <strong>observation</strong>. A variable contains all values that measure the same underlying attribute (like height, temperature, duration) across units. An observation contains all values measured on the same unit (like a person, or a day, or a race) across attributes.</p>

<p>A tidy version of the classroom data looks like this: (you’ll learn how the functions work a little later)</p>

In [4]:
library(tidyr)
library(dplyr)


Caricamento pacchetto: ‘dplyr’


I seguenti oggetti sono mascherati da ‘package:stats’:

    filter, lag


I seguenti oggetti sono mascherati da ‘package:base’:

    intersect, setdiff, setequal, union




In [5]:
classroom2 <- classroom |> 
  pivot_longer(quiz1:test1, names_to = "assessment", values_to = "grade") |> 
  arrange(name, assessment)

classroom2

name,assessment,grade
<chr>,<chr>,<chr>
Billy,quiz1,NA
Billy,quiz2,D
Billy,test1,C
Jenny,quiz1,A
Jenny,quiz2,A
Jenny,test1,B
Lionel,quiz1,B
Lionel,quiz2,C
Lionel,test1,B


<p>This makes the values, variables, and observations more clear. The dataset contains 36 values representing three variables and 12 observations. The variables are:</p>

<ol>
<li><p><code>name</code>, with four possible values (Billy, Suzy, Lionel, and Jenny).</p></li>
<li><p><code>assessment</code>, with three possible values (quiz1, quiz2, and test1).</p></li>
<li><p><code>grade</code>, with five or six values depending on how you think of the missing value (A, B, C, D, F, NA).</p></li>
</ol>

<p>The tidy data frame explicitly tells us the definition of an observation. In this classroom, every combination of <code>name</code> and <code>assessment</code> is a single measured observation. The dataset also informs us of missing values, which can and do have meaning. Billy was absent for the first quiz, but tried to salvage his grade. Suzy failed the first quiz, so she decided to drop the class. To calculate Billy’s final grade, we might replace this missing value with an F (or he might get a second chance to take the quiz). However, if we want to know the class average for Test 1, dropping Suzy’s structural missing value would be more appropriate than imputing a new value.</p>

<p>For a given dataset, it’s usually easy to figure out what are observations and what are variables, but it is surprisingly difficult to precisely define variables and observations in general. For example, if the columns in the classroom data were <code>height</code> and <code>weight</code> we would have been happy to call them variables. If the columns were <code>height</code> and <code>width</code>, it would be less clear cut, as we might think of height and width as values of a <code>dimension</code> variable. If the columns were <code>home phone</code> and <code>work phone</code>, we could treat these as two variables, but in a fraud detection environment we might want variables <code>phone number</code> and <code>number type</code> because the use of one phone number for multiple people might suggest fraud. A general rule of thumb is that it is easier to describe functional relationships between variables (e.g., <code>z</code> is a linear combination of <code>x</code> and <code>y</code>, <code>density</code> is the ratio of <code>weight</code> to <code>volume</code>) than between rows, and it is easier to make comparisons between groups of observations (e.g., average of group a vs.&nbsp;average of group b) than between groups of columns.</p>

<p>In a given analysis, there may be multiple levels of observation. For example, in a trial of new allergy medication we might have three observational types: demographic data collected from each person (<code>age</code>, <code>sex</code>, <code>race</code>), medical data collected from each person on each day (<code>number of sneezes</code>, <code>redness of eyes</code>), and meteorological data collected on each day (<code>temperature</code>, <code>pollen count</code>).</p>

<p>Variables may change over the course of analysis. Often the variables in the raw data are very fine grained, and may add extra modelling complexity for little explanatory gain. For example, many surveys ask variations on the same question to better get at an underlying trait. In early stages of analysis, variables correspond to questions. In later stages, you change focus to traits, computed by averaging together multiple questions. This considerably simplifies analysis because you don’t need a hierarchical model, and you can often pretend that the data is continuous, not discrete.</p>


## <h3 id="tidy-data">Tidy data</h3>

<p>Tidy data is a standard way of mapping the meaning of a dataset to its structure. A dataset is messy or tidy depending on how rows, columns and tables are matched up with observations, variables and types. In <strong>tidy data</strong>:</p>

<ol>
<li><p>Each variable is a column; each column is a variable.</p></li>
<li><p>Each observation is a row; each row is an observation.</p></li>
<li><p>Each value is a cell; each cell is a single value.</p></li>
</ol>

<p>This is Codd’s 3rd normal form, but with the constraints framed in statistical language, and the focus put on a single dataset rather than the many connected datasets common in relational databases.
<strong>Messy data</strong> is any other arrangement of the data.</p> <p>Tidy data makes it easy for an analyst or a computer to extract needed variables because it provides a standard way of structuring a dataset. Compare the different versions of the classroom data: in the messy version you need to use different strategies to extract different variables. This slows analysis and invites errors. If you consider how many data analysis operations involve all of the values in a variable (every aggregation function), you can see how important it is to extract
these values in a simple, standard way. Tidy data is particularly well suited for vectorised programming languages like R, because the layout ensures that values of different variables from the same observation are always paired.</p>

<p>While the order of variables and observations does not affect analysis, a good ordering makes it easier to scan the raw values. One way of organising variables is by their role in the analysis: are values fixed by the design of the data collection, or are they measured during the course of the experiment? Fixed variables describe the experimental design and are known in advance. Computer scientists often call fixed variables dimensions, and statisticians usually denote them with subscripts on random variables. Measured variables are what we actually
measure in the study. Fixed variables should come first, followed by measured variables, each ordered so that related variables are contiguous. Rows can then be ordered by the first variable, breaking ties with the second and subsequent (fixed) variables. This is the convention adopted by all tabular displays in this paper.</p>

# <h2 id="tidying">Tidying messy datasets</h2>

<p>Real datasets can, and often do, violate the three precepts of tidy data in almost every way imaginable. While occasionally you do get a dataset that you can start analysing immediately, this is the exception, not the rule. This section describes the five most common problems with messy datasets, along with their remedies:</p>

<ul>
<li><p>Column headers are values, not variable names.</p></li>
<li><p>Multiple variables are stored in one column.</p></li>
<li><p>Variables are stored in both rows and columns.</p></li>
<li><p>Multiple types of observational units are stored in the same table.</p></li>
<li><p>A single observational unit is stored in multiple tables.</p></li>
</ul>

<p>Surprisingly, most messy datasets, including types of messiness not explicitly described above, can be tidied with a small set of tools: pivoting (longer and wider) and separating. The following sections illustrate each problem with a real dataset that I have encountered, and show how to tidy them.</p>


## <h3 id="column-headers-are-values-not-variable-names">Column headers are values, not variable names</h3>

<p>A common type of messy dataset is tabular data designed for presentation, where variables form both the rows and columns, and column headers are values, not variable names. While I would call this arrangement messy, in some cases it can be extremely useful. It provides efficient storage for completely crossed designs, and it can lead to
extremely efficient computation if desired operations can be expressed as matrix operations.</p>

<p>The following code shows a subset of a typical dataset of this form. This dataset explores the relationship between income and religion in the US. It comes from a report produced by the Pew Research Center, an American think-tank that collects data on attitudes to topics ranging from religion to the internet, and produces many reports that contain datasets in this format.</p>

In [6]:
relig_income

religion,<$10k,$10-20k,$20-30k,$30-40k,$40-50k,$50-75k,$75-100k,$100-150k,>150k,Don't know/refused
<chr>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>
Agnostic,27,34,60,81,76,137,122,109,84,96
Atheist,12,27,37,52,35,70,73,59,74,76
Buddhist,27,21,30,34,33,58,62,39,53,54
Catholic,418,617,732,670,638,1116,949,792,633,1489
Don’t know/refused,15,14,15,11,10,35,21,17,18,116
Evangelical Prot,575,869,1064,982,881,1486,949,723,414,1529
Hindu,1,9,7,9,11,34,47,48,54,37
Historically Black Prot,228,244,236,238,197,223,131,81,78,339
Jehovah's Witness,20,27,24,24,21,30,15,11,6,37


<p>This dataset has three variables, <code>religion</code>, <code>income</code> and <code>frequency</code>. To tidy it, we need to <strong>pivot</strong> the non-variable columns into a two-column key-value pair. This action is often described as making a wide dataset longer (or taller).</p>

<p>When pivoting variables, we need to provide the name of the new key-value columns to create. After defining the columns to pivot (every column except for religion), you will need the name of the key column, which is the name of the variable defined by the values of the column headings. In this case, it’s <code>income</code>. The second argument is the name of the value column, <code>frequency</code>.</p>

In [7]:
relig_income |>
  pivot_longer(-religion, names_to = "income", values_to = "frequency")

religion,income,frequency
<chr>,<chr>,<dbl>
Agnostic,<$10k,27
Agnostic,$10-20k,34
Agnostic,$20-30k,60
Agnostic,$30-40k,81
Agnostic,$40-50k,76
Agnostic,$50-75k,137
Agnostic,$75-100k,122
Agnostic,$100-150k,109
Agnostic,>150k,84


<p>This form is tidy because each column represents a variable and each row represents an observation, in this case a demographic unit corresponding to a combination of <code>religion</code> and <code>income</code>.</p>

<p>This format is also used to record regularly spaced observations over time. For example, the Billboard dataset shown below records the date a song first entered the billboard top 100. It has variables for <code>artist</code>, <code>track</code>, <code>date.entered</code>, <code>rank</code> and <code>week</code>. The rank in each week after it enters the top 100 is recorded in 75 columns, <code>wk1</code> to <code>wk75</code>. This form of storage is not tidy, but it is useful for data entry. It reduces duplication since otherwise each song in each week would need its own row, and song metadata like title and artist would need to be repeated. This will be discussed in more depth in <a href="https://tidyr.tidyverse.org/articles/tidy-data.html#multiple-types">multiple types</a>.</p>

In [8]:
billboard

artist,track,date.entered,wk1,wk2,wk3,wk4,wk5,wk6,wk7,⋯,wk67,wk68,wk69,wk70,wk71,wk72,wk73,wk74,wk75,wk76
<chr>,<chr>,<date>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,⋯,<lgl>,<lgl>,<lgl>,<lgl>,<lgl>,<lgl>,<lgl>,<lgl>,<lgl>,<lgl>
2 Pac,Baby Don't Cry (Keep...,2000-02-26,87,82,72,77,87,94,99,⋯,NA,NA,NA,NA,NA,NA,NA,NA,NA,NA
2Ge+her,The Hardest Part Of ...,2000-09-02,91,87,92,NA,NA,NA,NA,⋯,NA,NA,NA,NA,NA,NA,NA,NA,NA,NA
3 Doors Down,Kryptonite,2000-04-08,81,70,68,67,66,57,54,⋯,NA,NA,NA,NA,NA,NA,NA,NA,NA,NA
3 Doors Down,Loser,2000-10-21,76,76,72,69,67,65,55,⋯,NA,NA,NA,NA,NA,NA,NA,NA,NA,NA
504 Boyz,Wobble Wobble,2000-04-15,57,34,25,17,17,31,36,⋯,NA,NA,NA,NA,NA,NA,NA,NA,NA,NA
98^0,Give Me Just One Nig...,2000-08-19,51,39,34,26,26,19,2,⋯,NA,NA,NA,NA,NA,NA,NA,NA,NA,NA
A*Teens,Dancing Queen,2000-07-08,97,97,96,95,100,NA,NA,⋯,NA,NA,NA,NA,NA,NA,NA,NA,NA,NA
Aaliyah,I Don't Wanna,2000-01-29,84,62,51,41,38,35,35,⋯,NA,NA,NA,NA,NA,NA,NA,NA,NA,NA
Aaliyah,Try Again,2000-03-18,59,53,38,28,21,18,16,⋯,NA,NA,NA,NA,NA,NA,NA,NA,NA,NA


<p>To tidy this dataset, we first use <code><a href="https://tidyr.tidyverse.org/reference/pivot_longer.html">pivot_longer()</a></code> to make the dataset longer. We transform the columns from <code>wk1</code> to <code>wk76</code>, making a new column for their names, <code>week</code>, and a new value for their values, <code>rank</code>:</p>

In [9]:
billboard2 <- billboard |> 
  pivot_longer(
    wk1:wk76, 
    names_to = "week", 
    values_to = "rank", 
    values_drop_na = TRUE
  )

billboard2

artist,track,date.entered,week,rank
<chr>,<chr>,<date>,<chr>,<dbl>
2 Pac,Baby Don't Cry (Keep...,2000-02-26,wk1,87
2 Pac,Baby Don't Cry (Keep...,2000-02-26,wk2,82
2 Pac,Baby Don't Cry (Keep...,2000-02-26,wk3,72
2 Pac,Baby Don't Cry (Keep...,2000-02-26,wk4,77
2 Pac,Baby Don't Cry (Keep...,2000-02-26,wk5,87
2 Pac,Baby Don't Cry (Keep...,2000-02-26,wk6,94
2 Pac,Baby Don't Cry (Keep...,2000-02-26,wk7,99
2Ge+her,The Hardest Part Of ...,2000-09-02,wk1,91
2Ge+her,The Hardest Part Of ...,2000-09-02,wk2,87


<p>Here we use <code>values_drop_na = TRUE</code> to drop any missing values from the rank column. In this data, missing values represent weeks that the song wasn’t in the charts, so can be safely dropped.</p>

<p>In this case it’s also nice to do a little cleaning, converting the week variable to a number, and figuring out the date corresponding to each week on the charts:</p>

In [10]:
billboard3 <- billboard2 |>
  mutate(
    week = as.integer(gsub("wk", "", week)),
    date = as.Date(date.entered) + 7 * (week - 1),
    date.entered = NULL
  )

billboard3

artist,track,week,rank,date
<chr>,<chr>,<int>,<dbl>,<date>
2 Pac,Baby Don't Cry (Keep...,1,87,2000-02-26
2 Pac,Baby Don't Cry (Keep...,2,82,2000-03-04
2 Pac,Baby Don't Cry (Keep...,3,72,2000-03-11
2 Pac,Baby Don't Cry (Keep...,4,77,2000-03-18
2 Pac,Baby Don't Cry (Keep...,5,87,2000-03-25
2 Pac,Baby Don't Cry (Keep...,6,94,2000-04-01
2 Pac,Baby Don't Cry (Keep...,7,99,2000-04-08
2Ge+her,The Hardest Part Of ...,1,91,2000-09-02
2Ge+her,The Hardest Part Of ...,2,87,2000-09-09


<p>Finally, it’s always a good idea to sort the data. We could do it by artist, track and week:</p>

In [11]:
billboard3 |> arrange(artist, track, week)

artist,track,week,rank,date
<chr>,<chr>,<int>,<dbl>,<date>
2 Pac,Baby Don't Cry (Keep...,1,87,2000-02-26
2 Pac,Baby Don't Cry (Keep...,2,82,2000-03-04
2 Pac,Baby Don't Cry (Keep...,3,72,2000-03-11
2 Pac,Baby Don't Cry (Keep...,4,77,2000-03-18
2 Pac,Baby Don't Cry (Keep...,5,87,2000-03-25
2 Pac,Baby Don't Cry (Keep...,6,94,2000-04-01
2 Pac,Baby Don't Cry (Keep...,7,99,2000-04-08
2Ge+her,The Hardest Part Of ...,1,91,2000-09-02
2Ge+her,The Hardest Part Of ...,2,87,2000-09-09


<p>Or by date and rank:</p>

In [12]:
billboard3 |> arrange(date, rank)

artist,track,week,rank,date
<chr>,<chr>,<int>,<dbl>,<date>
Lonestar,Amazed,1,81,1999-06-05
Lonestar,Amazed,2,54,1999-06-12
Lonestar,Amazed,3,44,1999-06-19
Lonestar,Amazed,4,39,1999-06-26
Lonestar,Amazed,5,38,1999-07-03
Lonestar,Amazed,6,33,1999-07-10
Lonestar,Amazed,7,29,1999-07-17
Amber,Sexual,1,99,1999-07-17
Lonestar,Amazed,8,29,1999-07-24


## <h3 id="multiple-variables-stored-in-one-column">Multiple variables stored in one column</h3>

<p>After pivoting columns, the key column is sometimes a combination of multiple underlying variable names. This happens in the <code>tb</code> (tuberculosis) dataset, shown below. This dataset comes from the World Health Organisation, and records the counts of confirmed tuberculosis cases by <code>country</code>, <code>year</code>, and demographic group.
The demographic groups are broken down by <code>sex</code> (m, f) and <code>age</code> (0-14, 15-25, 25-34, 35-44, 45-54, 55-64, unknown).</p>

In [13]:
tb <- as_tibble(read.csv("tb.csv", stringsAsFactors = FALSE))

In [14]:
tb

iso2,year,m04,m514,m014,m1524,m2534,m3544,m4554,m5564,⋯,f04,f514,f014,f1524,f2534,f3544,f4554,f5564,f65,fu
<chr>,<int>,<int>,<int>,<int>,<int>,<int>,<int>,<int>,<int>,⋯,<int>,<int>,<int>,<int>,<int>,<int>,<int>,<int>,<int>,<int>
AD,1989,NA,NA,NA,NA,NA,NA,NA,NA,⋯,NA,NA,NA,NA,NA,NA,NA,NA,NA,NA
AD,1990,NA,NA,NA,NA,NA,NA,NA,NA,⋯,NA,NA,NA,NA,NA,NA,NA,NA,NA,NA
AD,1991,NA,NA,NA,NA,NA,NA,NA,NA,⋯,NA,NA,NA,NA,NA,NA,NA,NA,NA,NA
AD,1992,NA,NA,NA,NA,NA,NA,NA,NA,⋯,NA,NA,NA,NA,NA,NA,NA,NA,NA,NA
AD,1993,NA,NA,NA,NA,NA,NA,NA,NA,⋯,NA,NA,NA,NA,NA,NA,NA,NA,NA,NA
AD,1994,NA,NA,NA,NA,NA,NA,NA,NA,⋯,NA,NA,NA,NA,NA,NA,NA,NA,NA,NA
AD,1996,NA,NA,0,0,0,4,1,0,⋯,NA,NA,0,1,1,0,0,1,0,NA
AD,1997,NA,NA,0,0,1,2,2,1,⋯,NA,NA,0,1,2,3,0,0,1,NA
AD,1998,NA,NA,0,0,0,1,0,0,⋯,NA,NA,NA,NA,NA,NA,NA,NA,NA,NA


<p>First we use <code><a href="https://tidyr.tidyverse.org/reference/pivot_longer.html">pivot_longer()</a></code> to gather up the non-variable columns:</p>

In [15]:
tb2 <- tb |> 
  pivot_longer(
    !c(iso2, year), 
    names_to = "demo", 
    values_to = "n", 
    values_drop_na = TRUE
  )
tb2

iso2,year,demo,n
<chr>,<int>,<chr>,<int>
AD,1996,m014,0
AD,1996,m1524,0
AD,1996,m2534,0
AD,1996,m3544,4
AD,1996,m4554,1
AD,1996,m5564,0
AD,1996,m65,0
AD,1996,f014,0
AD,1996,f1524,1


<p>Column headers in this format are often separated by a non-alphanumeric character (e.g.&nbsp;<code>.</code>, <code>-</code>, <code>_</code>, <code>:</code>), or have a fixed width format, like in this dataset. <code><a href="https://tidyr.tidyverse.org/reference/separate.html">separate()</a></code> makes it easy to split a compound variables into individual variables. You can either pass it a regular expression to split on (the default is to split on non alphanumeric columns), or a vector of character positions. In this case we want to split after the first character:</p>

In [16]:
tb3 <- tb2 |> 
  separate(demo, c("sex", "age"), 1)

tb3

iso2,year,sex,age,n
<chr>,<int>,<chr>,<chr>,<int>
AD,1996,m,014,0
AD,1996,m,1524,0
AD,1996,m,2534,0
AD,1996,m,3544,4
AD,1996,m,4554,1
AD,1996,m,5564,0
AD,1996,m,65,0
AD,1996,f,014,0
AD,1996,f,1524,1


<p>Storing the values in this form resolves a problem in the original data. We want to compare rates, not counts, which means we need to know the population. In the original format, there is no easy way to add a population variable. It has to be stored in a separate table, which makes it hard to correctly match populations to counts. In tidy form, adding variables for population and rate is easy because they’re just additional columns.</p>

<p>In this case, we could also do the transformation in a single step by supplying multiple column names to <code>names_to</code> and also supplying a grouped regular expression to <code>names_pattern</code>:</p>

In [17]:
tb |> pivot_longer(
  !c(iso2, year), 
  names_to = c("sex", "age"), 
  names_pattern = "(.)(.+)",
  values_to = "n", 
  values_drop_na = TRUE
)

iso2,year,sex,age,n
<chr>,<int>,<chr>,<chr>,<int>
AD,1996,m,014,0
AD,1996,m,1524,0
AD,1996,m,2534,0
AD,1996,m,3544,4
AD,1996,m,4554,1
AD,1996,m,5564,0
AD,1996,m,65,0
AD,1996,f,014,0
AD,1996,f,1524,1


## <h3 id="variables-are-stored-in-both-rows-and-columns">Variables are stored in both rows and columns</h3>

<p>The most complicated form of messy data occurs when variables are stored in both rows and columns. The code below loads daily weather data from the Global Historical Climatology Network for one weather station (MX17004) in Mexico for five months in 2010.</p>

In [18]:
weather <- as_tibble(read.csv("weather.csv", stringsAsFactors = FALSE))
weather

id,year,month,element,d1,d2,d3,d4,d5,d6,⋯,d22,d23,d24,d25,d26,d27,d28,d29,d30,d31
<chr>,<int>,<int>,<chr>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,⋯,<lgl>,<dbl>,<lgl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>
MX000017004201001TMAX,2010,1,tmax,NA,NA,NA,NA,NA,NA,⋯,NA,NA,NA,NA,NA,NA,NA,NA,27.8,NA
MX000017004201001TMIN,2010,1,tmin,NA,NA,NA,NA,NA,NA,⋯,NA,NA,NA,NA,NA,NA,NA,NA,14.5,NA
MX000017004201002TMAX,2010,2,tmax,NA,27.3,24.1,NA,NA,NA,⋯,NA,29.9,NA,NA,NA,NA,NA,NA,NA,NA
MX000017004201002TMIN,2010,2,tmin,NA,14.4,14.4,NA,NA,NA,⋯,NA,10.7,NA,NA,NA,NA,NA,NA,NA,NA
MX000017004201003TMAX,2010,3,tmax,NA,NA,NA,NA,32.1,NA,⋯,NA,NA,NA,NA,NA,NA,NA,NA,NA,NA
MX000017004201003TMIN,2010,3,tmin,NA,NA,NA,NA,14.2,NA,⋯,NA,NA,NA,NA,NA,NA,NA,NA,NA,NA
MX000017004201004TMAX,2010,4,tmax,NA,NA,NA,NA,NA,NA,⋯,NA,NA,NA,NA,NA,36.3,NA,NA,NA,NA
MX000017004201004TMIN,2010,4,tmin,NA,NA,NA,NA,NA,NA,⋯,NA,NA,NA,NA,NA,16.7,NA,NA,NA,NA
MX000017004201005TMAX,2010,5,tmax,NA,NA,NA,NA,NA,NA,⋯,NA,NA,NA,NA,NA,33.2,NA,NA,NA,NA


<p>It has variables in individual columns (<code>id</code>, <code>year</code>, <code>month</code>), spread across columns (<code>day</code>, d1-d31) and across rows (<code>tmin</code>, <code>tmax</code>) (minimum and maximum temperature). Months with fewer than 31 days have structural missing values for the last day(s) of the month.</p>

<p>To tidy this dataset we first use pivot_longer to gather the day columns:</p>

In [19]:
weather2 <- weather |>
  pivot_longer(
    d1:d31, 
    names_to = "day", 
    values_to = "value", 
    values_drop_na = TRUE
  )

weather2

id,year,month,element,day,value
<chr>,<int>,<int>,<chr>,<chr>,<dbl>
MX000017004201001TMAX,2010,1,tmax,d30,27.8
MX000017004201001TMIN,2010,1,tmin,d30,14.5
MX000017004201002TMAX,2010,2,tmax,d2,27.3
MX000017004201002TMAX,2010,2,tmax,d3,24.1
MX000017004201002TMAX,2010,2,tmax,d11,29.7
MX000017004201002TMAX,2010,2,tmax,d23,29.9
MX000017004201002TMIN,2010,2,tmin,d2,14.4
MX000017004201002TMIN,2010,2,tmin,d3,14.4
MX000017004201002TMIN,2010,2,tmin,d11,13.4


<p>For presentation, I’ve dropped the missing values, making them implicit rather than explicit. This is ok because we know how many days are in each month and can easily reconstruct the explicit missing values.</p>

<p>We’ll also do a little cleaning:</p>

In [28]:
weather3 <- weather2 |>
  mutate(day = as.integer(gsub("d", "", day))) |>
  mutate(id = gsub(".{3}$", "", id)) |>
  select(id, year, month, day, element, value)

weather3

id,year,month,day,element,value
<chr>,<int>,<int>,<int>,<chr>,<dbl>
MX000017004201001T,2010,1,30,tmax,27.8
MX000017004201001T,2010,1,30,tmin,14.5
MX000017004201002T,2010,2,2,tmax,27.3
MX000017004201002T,2010,2,3,tmax,24.1
MX000017004201002T,2010,2,11,tmax,29.7
MX000017004201002T,2010,2,23,tmax,29.9
MX000017004201002T,2010,2,2,tmin,14.4
MX000017004201002T,2010,2,3,tmin,14.4
MX000017004201002T,2010,2,11,tmin,13.4


<p>This dataset is mostly tidy, but the <code>element</code> column is not a variable; it stores the names of variables. (Not shown in this example are the other meteorological variables <code>prcp</code> (precipitation) and <code>snow</code> (snowfall)). Fixing this requires widening the data: <code><a href="../reference/pivot_wider.html">pivot_wider()</a></code> is inverse of <code><a href="../reference/pivot_longer.html">pivot_longer()</a></code>, pivoting <code>element</code> and <code>value</code> back out across multiple columns:</p>

In [29]:
weather3 |>
  pivot_wider(names_from = element, values_from = value)

id,year,month,day,tmax,tmin
<chr>,<int>,<int>,<int>,<dbl>,<dbl>
MX000017004201001T,2010,1,30,27.8,14.5
MX000017004201002T,2010,2,2,27.3,14.4
MX000017004201002T,2010,2,3,24.1,14.4
MX000017004201002T,2010,2,11,29.7,13.4
MX000017004201002T,2010,2,23,29.9,10.7
MX000017004201003T,2010,3,5,32.1,14.2
MX000017004201003T,2010,3,10,34.5,16.8
MX000017004201003T,2010,3,16,31.1,17.6
MX000017004201004T,2010,4,27,36.3,16.7


<p>This form is tidy: there’s one variable in each column, and each row represents one day.</p>

## <h3 id="multiple-types">Multiple types in one table</h3>

<p>Datasets often involve values collected at multiple levels, on different types of observational units. During tidying, each type of observational unit should be stored in its own table. This is closely related to the idea of database normalisation, where each fact is expressed in only one place. It’s important because otherwise inconsistencies can arise.</p>

<p>The billboard dataset actually contains observations on two types of observational units: the song and its rank in each week. This manifests itself through the duplication of facts about the song: <code>artist</code> is repeated many times.</p>

<p>This dataset needs to be broken down into two pieces: a song dataset which stores <code>artist</code> and <code>song name</code>, and a ranking dataset which gives the <code>rank</code> of the <code>song</code> in each <code>week</code>. We first extract a <code>song</code> dataset:</p>

In [33]:
song <- billboard3 |>
  distinct(artist, track) |>
  mutate(song_id = row_number())

song

artist,track,song_id
<chr>,<chr>,<int>
2 Pac,Baby Don't Cry (Keep...,1
2Ge+her,The Hardest Part Of ...,2
3 Doors Down,Kryptonite,3
3 Doors Down,Loser,4
504 Boyz,Wobble Wobble,5
98^0,Give Me Just One Nig...,6
A*Teens,Dancing Queen,7
Aaliyah,I Don't Wanna,8
Aaliyah,Try Again,9


<p>Then use that to make a <code>rank</code> dataset by replacing repeated song facts with a pointer to song details (a unique song id):</p>

In [34]:
rank <- billboard3 |>
  left_join(song, c("artist", "track")) |>
  select(song_id, date, week, rank)

rank

song_id,date,week,rank
<int>,<date>,<int>,<dbl>
1,2000-02-26,1,87
1,2000-03-04,2,82
1,2000-03-11,3,72
1,2000-03-18,4,77
1,2000-03-25,5,87
1,2000-04-01,6,94
1,2000-04-08,7,99
2,2000-09-02,1,91
2,2000-09-09,2,87


<p>You could also imagine a <code>week</code> dataset which would record background information about the week, maybe the total number of songs sold or similar “demographic” information.</p>

<p>Normalisation is useful for tidying and eliminating inconsistencies. However, there are few data analysis tools that work directly with relational data, so analysis usually also requires denormalisation or the merging the datasets back into one table.</p>

## <h3 id="one-type-in-multiple-tables">One type in multiple tables</h3>

<p>It’s also common to find data values about a single type of observational unit spread out over multiple tables or files. These tables and files are often split up by another variable, so that each represents a single year, person, or location. As long as the format for individual records is consistent, this is an easy problem to fix:</p>

<ol>
<li><p>Read the files into a list of tables.</p></li>
<li><p>For each table, add a new column that records the original file name (the file name is often the value of an important variable).</p></li>
<li><p>Combine all tables into a single table.</p></li>
</ol>

<p>Purrr makes this straightforward in R. The following code generates a vector of file names in a directory (<code>data/</code>) which match a regular expression (ends in <code>.csv</code>). Next we name each element of the vector with the name of the file. We do this because will preserve the names in the following step, ensuring that each row in the final data frame is labeled with its source. Finally, <code><a href="https://purrr.tidyverse.org/reference/map_dfr.html">map_dfr()</a></code> loops over each path, reading in the csv file and
combining the results into a single data frame.</p>

In [39]:
library(purrr)

paths <- dir("data", pattern = "\\.csv$", full.names = TRUE)
names(paths) <- basename(paths)
map_dfr(paths, read.csv, stringsAsFactors = FALSE, .id = "filename")

<0 x 0 matrix>

<p>Once you have a single table, you can perform additional tidying as needed. An example of this type of cleaning can be found at <a href="https://github.com/hadley/data-baby-names">https://github.com/hadley/data-baby-names</a> which takes 129 yearly baby name tables provided by the US Social Security Administration and combines them into a single file.</p>

<p>A more complicated situation occurs when the dataset structure changes over time. For example, the datasets may contain different variables, the same variables with different names, different file formats, or different conventions for missing values. This may require you to tidy each file to individually (or, if you’re lucky, in small groups) and then combine them once tidied. An example of this type of tidying is illustrated in <a href="https://github.com/hadley/data-fuel-economy">https://github.com/hadley/data-fuel-economy</a>, which shows the tidying of <span>epa</span> fuel economy data for over 50,000 cars from 1978 to 2008. The raw data is available online, but each year is stored in a separate file and there are four major formats with many minor variations, making tidying this dataset a considerable challenge.</p>